In [5]:
import sys
import os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)


import importlib
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import DDPG
import matplotlib.pyplot as plt
import torch

# Import as module
import environment.users.citation_users_loader as citation_users_loader
importlib.reload(citation_users_loader)

import environment.citations.citation_selector as citation_selector
importlib.reload(citation_selector)

import environment.citation_reward_shaping as citation_reward_shaping
importlib.reload(citation_reward_shaping)

import environment.citations.citation as citation
importlib.reload(citation)

import environment.users.citation_user as citation_user
importlib.reload(citation_user)

import environment.citations.citation_loader as citation_loader
importlib.reload(citation_loader)

import environment.citation_memory as citation_memory
importlib.reload(citation_memory)

import environment.users.user_rater as user_rater
importlib.reload(user_rater)

import environment.user_env as user_env
importlib.reload(user_env)

import environment.user_env_continuous_wrapper as user_env_continuous_wrapper
importlib.reload(user_env_continuous_wrapper)

from environment.user_env import UserEnv
from environment.users.citation_users_loader import CitationUsersLoader
from environment.citations.citation_loader import CitationsLoader
from environment.citations.citation import Citation
from environment.citations.citation_selector import CitationsSelector
from environment.citation_memory import CitationMemory
from environment.users.citation_user import CitationUser
from environment.users.user_rater import RuleBasedCitationRater
from environment.citation_reward_shaping import CitationRewardReshapingExpDecay
from environment.user_env_continuous_wrapper import UserEnvContinuousWrapper

In [6]:
users_loader = CitationUsersLoader("../environment/users/datasets/users_with_popular_interests.json")
user_list = users_loader.get_users()
paper_loader = CitationsLoader("../environment/citations/datasets/cleaned-scientometrics-and-bibliometrics-research.csv")
selector = CitationsSelector()
rater = RuleBasedCitationRater()
reward_shaping = CitationRewardReshapingExpDecay()

In [9]:
user_rewards = []
timesteps = 30000

for user in user_list:
    env = UserEnv(
        render_mode= "human",
        items_loader=paper_loader,
        user_list=[user],
        items_selector=selector,
        reward_shaping=reward_shaping,
        render_path="./tmp/render/",
        debug=True,
        debug_path=f"debug_log/user_env_DDPG_user{user.id}_{timesteps}.log",
    )
    env = Monitor(env, filename=f"monitor_DDPG_user{user.id}.csv", info_keywords=("user_id", "item_interact", "item_clicks"))
    env = UserEnvContinuousWrapper(env)

    model = DDPG(
        policy="MultiInputPolicy",        # fully connected network
        env=env,
        batch_size=64,
        buffer_size=1_000,
        learning_rate=1e-3,
        tau=0.005,                 # for soft target updates
        gamma=0.99,
        verbose=1,
        device="cpu",              # or "cuda"
    )

    model.learn(total_timesteps=timesteps)
    user_rewards.append(env.get_episode_rewards())


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:16                                                                                   │
│                                                                                                  │
│   13 │   │   debug_path=f"debug_log/user_env_DDPG_user{user.id}_{timesteps}.log",                │
│   14 │   )                                                                                       │
│   15 │   env = Monitor(env, filename=f"monitor_DDPG_user{user.id}.csv", info_keywords=("user_    │
│ ❱ 16 │   env = UserEnvContinuousWrapper(env)                                                     │
│   17 │                                                                                           │
│   18 │   model = DDPG(                                                                           │
│   19 │   │   policy="MultiInputPolicy",        # fully connected network                         │
│                                                                                                  │
│ /Users/monika/SUBER/environment/user_env_continuous_wrapper.py:16 in __init__                    │
│                                                                                                  │
│   13 │   │   vecs = []                                                                           │
│   14 │   │   for item in items:                                                                  │
│   15 │   │   │   # topic embedding                                                               │
│ ❱ 16 │   │   │   t = env.bert_model.encode(item.topics).mean(axis=0).astype(np.float32)          │
│   17 │   │   │   # u = env.bert_model.encode(env._user.interests).mean(axis=0).astype(np.floa    │
│   18 │   │   │   # normalized year & citation already in [0,1]                                   │
│   19 │   │   │   y = np.array([item.quartile_year],  dtype=np.float32)                           │
│                                                                                                  │
│ /Users/monika/SUBER/.venv/lib/python3.9/site-packages/sentence_transformers/SentenceTransformer. │
│ py:685 in encode                                                                                 │
│                                                                                                  │
│    682 │   │   │   features.update(extra_features)                                               │
│    683 │   │   │                                                                                 │
│    684 │   │   │   with torch.no_grad():                                                         │
│ ❱  685 │   │   │   │   out_features = self.forward(features, **kwargs)                           │
│    686 │   │   │   │   if self.device.type == "hpu":                                             │
│    687 │   │   │   │   │   out_features = copy.deepcopy(out_features)                            │
│    688                                                                                           │
│                                                                                                  │
│ /Users/monika/SUBER/.venv/lib/python3.9/site-packages/sentence_transformers/SentenceTransformer. │
│ py:758 in forward                                                                                │
│                                                                                                  │
│    755 │   │   for module_name, module in self.named_children():                                 │
│    756 │   │   │   module_kwarg_keys = self.module_kwargs.get(module_name, [])                   │
│    757 │   │   │   module_kwargs = {key: value for key, value in kwargs.items() if key in modul  │
│ ❱  758 │   │   │   input = module(input, **module_kwargs)                                        │
│    759 │   │   return input                                

In [ ]:
def plot_rewards(all_user_rewards, window_size=40):
    """
    all_user_rewards: list of lists/arrays, each one is the rewards for one user
    window_size: smoothing window for moving average
    """
    plt.xlabel('Episode')
    plt.ylabel('Reward')

    for i, rewards in enumerate(all_user_rewards):
        rewards_t = torch.tensor(rewards, dtype=torch.float)

        # raw rewards (optional)
        # plt.plot(rewards_t.numpy(), alpha=0.2, label=f'run {i} raw')

        # smoothed (moving average)
        if len(rewards_t) >= window_size:
            means = rewards_t.unfold(0, window_size, 1).mean(1).view(-1)
            # pad the beginning so lengths match
            pad = torch.zeros(window_size - 1)
            smoothed = torch.cat((pad, means))
            plt.plot(smoothed.numpy(), label=f'User {i}')
        else:
            # too short to smooth, just plot raw
            plt.plot(rewards_t.numpy(), label=f'User {i}')

    plt.legend()
    plt.show()

plot_rewards(user_rewards)